In [15]:
# imports for langchain and Chroma and plotly
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

In [26]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter

# Load PDF
loader = PyPDFLoader("vedenta.pdf")
documents = loader.load()

print(len(documents), "pages loaded")

360 pages loaded


# 🪓 What is `RecursiveCharacterTextSplitter`?

## 🚨 The Problem
If you have a big text (like a PDF page or a book), you can’t throw it as-is into an LLM.  
You need to **break it into smaller chunks**.  
But you don’t want chunks to break in the middle of a sentence or word.

---

## ✅ What it Does
`RecursiveCharacterTextSplitter` tries to split text **nicely** by looking for natural breakpoints.  

It has a priority order of separators, usually like:
1. Paragraphs (`\n\n`)  
2. Sentences (`\n`)  
3. Spaces (`" "`)  
4. If nothing works → just chop at character length  

It **recursively** applies this logic until the text fits into your chunk size.  

---

## 📖 Example
```python
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100, 
    chunk_overlap=20
)

text = """Artificial Intelligence (AI) is transforming the world. 
It is used in healthcare, finance, and education."""

chunks = text_splitter.split_text(text)
print(chunks)
```

**Result:**
```text
[
  "Artificial Intelligence (AI) is transforming the world.",
  "It is used in healthcare, finance, and education."
]
```

---

## 🤔 Why It’s Better
- A plain splitter would cut anywhere (even in the middle of a word).  
- Recursive splitter makes sure the cuts are as **natural** as possible.  

---

## ⚡ TL;DR
`RecursiveCharacterTextSplitter` = **Smart text chopper**  
It breaks text into **LLM-friendly chunks** while keeping them **human-readable**.  


In [32]:
# Chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)

print("Before:", len(documents))         # e.g. 5 pages
print(len(chunks), "chunks created")
print(chunks[0].page_content[:300])  # preview first chunk 300 chars
#print(chunks[0])


Before: 360
2831 chunks created
V
EDL/Sec./SE/25-26/50   June 18, 2025 
BS
E Limited    National Stock Exchange of India Limited 
Phiroze Jeejeebhoy Towers   “Exchange Plaza” 5th Floor Plot No., C/l, G Block 
Dalal Street, Fort   Bandra-Kurla Complex, Bandra (East), 
Mumbai - 400 001   Mumbai – 400 051 
S
crip Code: 500295   Scrip


> Entering new ConversationalRetrievalChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: what are the business vedenta is involved into?
Assistant: Vedanta is involved in a variety of businesses across the natural resources and technology sectors. Specifically, they operate in the following areas:

1. Oil & Gas
2. Zinc
3. Lead
4. Silver
5. Copper
6. Iron Ore
7. Steel
8. Nickel
9. Aluminium
10. Power
11. Glass Substrate
12. Electronics

Additionally, Vedanta focuses on

In [8]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [9]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [10]:
# Put the chunks of data into a Vector Store that associates a Vector Embedding with each chunk

embeddings = OpenAIEmbeddings()

# If you would rather use the free Vector Embeddings from HuggingFace sentence-transformers
# Then replace embeddings = OpenAIEmbeddings()
# with:
# from langchain.embeddings import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [11]:
# Check if a Chroma Datastore already exists - if so, delete the collection to start from scratch

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [12]:
# Create our Chroma vectorstore!

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 2831 documents


In [13]:
# Get one vector and find how many dimensions it has

collection = vectorstore._collection
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"The vectors have {dimensions:,} dimensions")

The vectors have 1,536 dimensions


In [20]:
from langchain_core.callbacks import StdOutCallbackHandler

# create a new Chat with OpenAI
llm = ChatOpenAI(temperature=0.7, model_name=MODEL)

# set up the conversation memory for the chat
memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)

# the retriever is an abstraction over the VectorStore that will be used during RAG
retriever = vectorstore.as_retriever()

# putting it together: set up the conversation chain with the GPT 4o-mini LLM, the vector store and memory
conversation_chain = ConversationalRetrievalChain.from_llm(llm=llm, retriever=retriever, memory=memory, callbacks=[StdOutCallbackHandler()])

In [21]:
# Wrapping in a function - note that history isn't used, as the memory is in the conversation_chain

def chat(message, history):
    result = conversation_chain.invoke({"question": message})
    return result["answer"]

In [22]:
# And in Gradio:

view = gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.




> Entering new ConversationalRetrievalChain chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
and lead, catering to a wide range of industrial consumers. Vedanta primarily serves industries such as automotive, 
steel, power generation, infrastructure, battery manufacturing, and oil, reinforcing its role as a key supplier in the 
global commodities market.
STATUTORY REPORTSCOFS
346 347
Business Responsibility & Sustainability ReportVEDANTA LIMITED
Integrated Report and Annual Accounts 2024-25 Integrated Report and Annual Accounts 2024-25

over the last few decades has been nothing short of 
remarkable. From our origins as a single-asset business, 
we have evolved into a globally diversified natural 
resources group with strategic interests 